# Multi-Agent RAG Educational Pipeline With Fine-Tuned Model

**Author:** asket  
**Location:** `week8/community_contributions/asket/`

This notebook implements a **Multi-Agent RAG Educational Pipeline with fine-tuned model** support. Three-stage flow for **university students only** (course questions, assignments, exam prep). Inspired by **[Klingbo.com](https://klingbo.com)** — Ghana's AI-powered education platform.

**Fine-tuned model:** The Query Expander can run on Modal with a fine-tuned model (e.g. education/legal QA); optional deploy. The Student Advisor uses litellm/OpenAI; the pipeline is designed to plug in fine-tuned models for expander or tutor.  

**For university students only:** assignment help, exam revision, course content — no legal or non-academic use.  

**Pipeline stages:**  
1. **Query Expander** — turns one course question into 4–5 search keyphrases (local or Modal + fine-tuned model).  
2. **Academic Researcher** — RAG over course materials (Chroma).  
3. **Student Advisor** — final answer from retrieved context (litellm/OpenAI or fine-tuned).  

**Run from `week8/`** so that the asket pipeline resolves correctly.

## Setup — run from week8/

Set working directory to **week8** and add **asket** to the path so we can import `student_pipeline`.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "agents").exists() and (cwd / "deal_agent_framework.py").exists():
    week8_dir = cwd
elif (cwd.parent.parent / "agents").exists():
    week8_dir = cwd.parent.parent
else:
    week8_dir = cwd
if str(week8_dir) not in sys.path:
    sys.path.insert(0, str(week8_dir))
sys.path = [str(week8_dir)] + [p for p in sys.path if p != str(week8_dir) and "community_contributions/asket" not in p]
asket_dir = week8_dir / "community_contributions" / "asket"
if asket_dir.exists() and str(asket_dir) not in sys.path:
    sys.path.insert(0, str(asket_dir))
os.chdir(week8_dir)
print("Working directory:", os.getcwd())

from dotenv import load_dotenv
load_dotenv(override=True)
import logging
logging.getLogger().setLevel(logging.INFO)

### Install dependencies (run once)

Run this cell once if you see missing-module errors; then **Kernel → Restart** and re-run from Setup.

In [ ]:
%pip install -q litellm chromadb gradio modal openai python-dotenv sentence-transformers --user --break-system-packages

## Build the RAG index

Index the **knowledge_base** (e.g. `knowledge_base/education_sample.txt`) into Chroma. The sample includes calculus, contract law, study skills, and WASSCE/university prep — Klingbo-style content.

In [ ]:
import student_pipeline

client = student_pipeline.build_index(force=False)
coll = client.get_collection(student_pipeline.COLLECTION_NAME)
print("Chunks in index:", coll.count())

## Run the pipeline once

Ask a question. You'll see **thinking** (expander → researcher → tutor) and the **final answer**. Set `use_modal_expander=True` if you've deployed `modal_student_expander.py` to Modal.

In [ ]:
import importlib
import student_pipeline
importlib.reload(student_pipeline)  # ensure latest run_pipeline (returns 4 values)

question = "What is consideration in contract law?"
use_modal = False  # Set True after: modal deploy -m community_contributions/asket/modal_student_expander.py

expander_out, researcher_out, answer, status = student_pipeline.run_pipeline(question, use_modal_expander=use_modal, client=client)
print("Status:", status)
print("\n=== Agent 1: Query Expander (search keyphrases) ===")
print(expander_out)
print("\n=== Agent 2: Academic Researcher (retrieved course materials) ===")
print(researcher_out[:800] + "..." if len(researcher_out) > 800 else researcher_out)
print("\n=== Agent 3: Student Advisor (final answer) ===")
print(answer)

## Gradio UI — Multi-Agent RAG Educational Pipeline With Fine-Tuned Model

Three-panel interface: **Query Expander** (optional fine-tuned model on Modal) → **Academic Researcher** (RAG) → **Student Advisor**. For university course questions only. Requires **OPENAI_API_KEY** (or litellm config) for the Student Advisor step.

In [ ]:
import gradio as gr
import student_pipeline

_client = student_pipeline.build_index()

def run_agents(question: str, use_modal: bool, progress=gr.Progress()):
    if not question or not question.strip():
        return "", "", "", "Enter a course question."
    progress(0.2, desc="Running pipeline…")
    expander_out, researcher_out, answer, status = student_pipeline.run_pipeline(question, use_modal_expander=use_modal, client=_client)
    return expander_out, researcher_out, answer, f"System Status: {status}"

with gr.Blocks(title="Multi-Agent RAG Educational Pipeline With Fine-Tuned Model", theme=gr.themes.Soft()) as ui:
    gr.Markdown("## Multi-Agent RAG Educational Pipeline With Fine-Tuned Model")
    gr.Markdown("**For university students only.** Course questions, assignments, exam prep. RAG + optional fine-tuned model (Modal). Inspired by [Klingbo](https://klingbo.com).")
    status_out = gr.Textbox(label="System Status", value="Ready", interactive=False)
    inp = gr.Textbox(label="Enter your course question:", placeholder="e.g. What is consideration in contract law?")
    use_modal = gr.Checkbox(label="Use Modal query expander", value=False)
    run_btn = gr.Button("Run Agents")
    gr.Markdown("---")
    with gr.Row():
        expander_out = gr.Textbox(label="Agent 1: Query Expander — Search keyphrases", lines=6, interactive=False)
    with gr.Row():
        researcher_out = gr.Textbox(label="Agent 2: Academic Researcher — Retrieved course materials", lines=8, interactive=False)
    with gr.Row():
        answer_out = gr.Textbox(label="Agent 3: Student Advisor — Final answer", lines=8, interactive=False)
    run_btn.click(fn=run_agents, inputs=[inp, use_modal], outputs=[expander_out, researcher_out, answer_out, status_out])

ui.launch()

---

## Summary

- **Query Expander:** one course question → 4–5 search keyphrases (local or Modal with fine-tuned model).  
- **Academic Researcher:** Chroma RAG over `knowledge_base/` (course materials only).  
- **Student Advisor:** litellm/OpenAI (or fine-tuned model) generates the final answer for university students.  
- **Gradio UI:** three-panel layout (Expander → Researcher → Student Advisor). **Multi-Agent RAG Educational Pipeline With Fine-Tuned Model.** University students only.  

**Optional:** Deploy the expander to Modal:  
`modal deploy -m community_contributions/asket/modal_student_expander.py` from **week8/**.